In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", color_codes=True)

from collections import Counter
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics.cluster import adjusted_mutual_info_score, contingency_matrix
from weasyprint import HTML

from pysrc.papers.pubtrends_config import AnalyzerSettings

from utils.analysis import align_clusterings_for_sklearn, rebuild_similarity_graph, \
                           recalculate_topic_analysis, get_direct_references_subgraph
from utils.io import load_analyzer, load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [ ]:
RESULTS_JSON_ORIGINAL = '/home/willenjoy/work/pubtrends/local/nature_reviews/grid_search/full_2021_03_03/grid_search_full_2021-03-03_13_22.json' 

In [ ]:
with open(RESULTS_JSON_ORIGINAL, 'r') as f:
    results = json.load(f)

In [ ]:
# bad clustering
# PMID = 29321682  
# BEST_SOI = (1, 0.0, 8.0, 0.0)
# LEVEL = 1

# good clustering
PMID = 27904142  
BEST_SOI = (1, 0.0625, 0.125, 1.0)
LEVEL = 1

In [ ]:
ground_truth = {}
clustering = load_clustering(PMID)
for level in range(1, get_clustering_level(clustering)):
    ground_truth[(PMID, level)] = preprocess_clustering(clustering, max_level=level,
                                                        include_box_sections=False,
                                                        uniqueness_method='unique_only')

In [ ]:
param_names = ['SIMILARITY_COCITATION',
               'SIMILARITY_BIBLIOGRAPHIC_COUPLING',
               'SIMILARITY_CITATION',
               'SIMILARITY_TEXT_CITATION']

def get_best_partition(analyzer, subgraph, soi):
    # Turn off topic merge
    params = dict(
        TOPIC_MIN_SIZE=0,
        TOPICS_MAX_NUMBER=500
    )
    for name, value in zip(param_names, soi):
        params[name] = value
    settings = AnalyzerSettings(**params)
    partition = recalculate_topic_analysis(analyzer, 
                                           graph=subgraph,
                                           settings=settings)
    return partition

In [ ]:
analyzer = load_analyzer(PMID)
rebuild_similarity_graph(analyzer)
subgraph = get_direct_references_subgraph(analyzer, str(PMID))

In [ ]:
# Should set similarity values that led to the best clustering
best_partition = get_best_partition(analyzer, subgraph, BEST_SOI)

In [ ]:
gt_partition = ground_truth[(PMID, LEVEL)]

In [ ]:
def plot_contingency_matrix(cm, title_str):
    fig = plt.figure(figsize = (10,7))
    sns.heatmap(cm, annot=True)
    plt.title(title_str)
    plt.xlabel('PubTrends')
    plt.ylabel('Nature Reviews')

In [ ]:
def get_param_contingency(analyzer, gt_partition, param, include_outside=False, normalize=False):
    n_gt_clusters = len(set(gt_partition.values()))
    
    # Calculate number of possible edges for given cluster sizes (normalize weights later)
    gt_sizes = list(Counter(gt_partition.values()).values())
    expected_edges = np.zeros((n_gt_clusters, n_gt_clusters))
    for i in range(n_gt_clusters):
        for j in range(n_gt_clusters):
            if i == j:
                expected_edges[i, j] = gt_sizes[i] * (gt_sizes[i] - 1) / 2
            else:
                expected_edges[i, j] = gt_sizes[i] * gt_sizes[j]
    
    if include_outside:
        param_contingency = np.zeros((n_gt_clusters + 1, n_gt_clusters + 1))
    else:
        param_contingency = np.zeros((n_gt_clusters, n_gt_clusters))
    for s, e, data in analyzer.similarity_graph.edges(data=True):
        if param in data:
            s_cluster = gt_partition[s] if s in gt_partition else n_gt_clusters
            e_cluster = gt_partition[e] if e in gt_partition else n_gt_clusters
            if include_outside:
                param_contingency[s_cluster, e_cluster] += data[param]
                param_contingency[e_cluster, s_cluster] += data[param]
            elif s_cluster < n_gt_clusters and e_cluster < n_gt_clusters:
                param_contingency[s_cluster, e_cluster] += data[param]
                param_contingency[e_cluster, s_cluster] += data[param]
                
    if normalize:
        return param_contingency / expected_edges
    else:
        return param_contingency

In [ ]:
fig = plt.figure(figsize=(18, 12))

for i, param in enumerate(['cocitation', 'bibcoupling', 'citation', 'text', 'similarity', 'contingency']):
    ax = fig.add_subplot(2, 3, i + 1)
    if param == 'contingency':
        contingency = contingency_matrix(*align_clusterings_for_sklearn(best_partition, gt_partition))
        sns.heatmap(contingency, ax=ax, annot=True)
        ax.set_title('Contingency Matrix')
        ax.set_xlabel('PubTrends')
        ax.set_ylabel('Nature Reviews')
    else:
        contingency = get_param_contingency(analyzer, gt_partition, param, normalize=True)
        sns.heatmap(contingency, ax=ax, annot=True, fmt='.2f')
        ax.set_title(param)